# LeetCode 3173 — Bitwise OR of Adjacent Elements bằng Python

Link đề: https://leetcode.com/problems/bitwise-or-of-adjacent-elements/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation, Array

Skeleton LeetCode:

```python
class Solution:
    def orArray(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản Premium để xem đề trên LeetCode). Đề trong notebook được viết theo hiểu biết về bài, nên **đối chiếu lại đề gốc nếu có tài khoản Premium** — đặc biệt phần ràng buộc (`2 <= nums.length <= 100`, `0 <= nums[i] <= 100`) và ba ví dụ bên dưới. Mọi đáp án trong notebook đều đã được kiểm chứng bằng code.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng `nums` độ dài `n`, tạo mảng `answer` độ dài **`n - 1`** với `answer[i] = nums[i] | nums[i + 1]`, trong đó `|` là phép **OR theo bit**.
2. Nắm các công cụ Python: toán tử **`|`** (OR bit) khác hẳn **`or`** (logic) và khác **`+`**, biểu diễn nhị phân `bin(x)` / `f"{x:08b}"`, **`zip(nums, nums[1:])`**, list comprehension, `itertools.pairwise`, `map(operator.or_, ...)`.
3. Đi từ cách **tự cộng OR từng bit** (để hiểu `|` làm gì) → **một vòng lặp `append`** → **list comprehension theo chỉ số** → **`zip` ghép cặp kề nhau** → **`map` / `pairwise`**.
4. Hiểu mấu chốt: mỗi phần tử kết quả chỉ phụ thuộc **hai phần tử đứng cạnh nhau** nên các phép tính **độc lập**; mảng kết quả có `n - 1` phần tử vì có `n - 1` cặp kề nhau.
5. Tránh các lỗi kinh điển: dùng **`or`** thay cho `|` (ra một trong hai số chứ không phải OR bit), dùng **`+`**, tạo kết quả dài **`n`** (hoặc `IndexError` vì `range(len(nums))`), và sửa mảng **tại chỗ theo hướng sai** làm hỏng dữ liệu còn cần dùng.

## 2. Đề bài & ví dụ

Cho mảng `nums` có độ dài `n`. Trả về mảng `answer` có độ dài **`n - 1`** sao cho

`answer[i] = nums[i] | nums[i + 1]`

trong đó `|` là phép **OR theo bit (bitwise OR)**.

**Ví dụ 1**

- Input: `nums = [1,3,7,15]`
- Output: `[3,7,15]`
- Giải thích: `1 | 3 = 3`, `3 | 7 = 7`, `7 | 15 = 15`.

**Ví dụ 2**

- Input: `nums = [8,4,2]`
- Output: `[12,6]`
- Giải thích: `8 | 4 = 12` (`1000 | 0100 = 1100`), `4 | 2 = 6` (`0100 | 0010 = 0110`).

**Ví dụ 3**

- Input: `nums = [5,4,9,11]`
- Output: `[5,13,11]`
- Giải thích: `5 | 4 = 5` (`0101 | 0100 = 0101`), `4 | 9 = 13` (`0100 | 1001 = 1101`), `9 | 11 = 11` (`1001 | 1011 = 1011`).

**Ràng buộc:** `2 <= nums.length <= 100`, `0 <= nums[i] <= 100`.

(Bài Premium: nên đối chiếu lại đề gốc nếu có tài khoản Premium.)

In [1]:
from typing import List
from itertools import pairwise
from operator import or_
import random
import itertools

examples = [
    ([1, 3, 7, 15], [3, 7, 15]),
    ([8, 4, 2], [12, 6]),
    ([5, 4, 9, 11], [5, 13, 11]),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 3, 7, 15] -> kỳ vọng [3, 7, 15]
[8, 4, 2] -> kỳ vọng [12, 6]
[5, 4, 9, 11] -> kỳ vọng [5, 13, 11]


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân của một số

Máy tính lưu số nguyên bằng các **bit** `0/1`. Mỗi bit ứng với một lũy thừa của `2`: `13 = 8 + 4 + 1 = 1101` (nhị phân). `bin(x)` trả về chuỗi có tiền tố `0b`; `f"{x:08b}"` in `x` thành `8` bit, thêm số `0` phía trước cho đều.

In [2]:
for x in [1, 3, 5, 9, 13, 15]:
    print(f"{x:>2} = {x:08b} (bin: {bin(x)})")
print(int("1101", 2))       # 13 : đổi ngược chuỗi nhị phân về số

 1 = 00000001 (bin: 0b1)
 3 = 00000011 (bin: 0b11)
 5 = 00000101 (bin: 0b101)
 9 = 00001001 (bin: 0b1001)
13 = 00001101 (bin: 0b1101)
15 = 00001111 (bin: 0b1111)
13


### 3.2. Phép OR theo bit `|`

`a | b` thực hiện **OR trên từng cặp bit cùng vị trí**: bit kết quả bằng `1` nếu **ít nhất một** trong hai bit là `1`; chỉ bằng `0` khi **cả hai** đều `0`.

| bit `a` | bit `b` | `a \| b` |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

In [3]:
for a, b in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(f"{a} | {b} = {a | b}")
a, b = 4, 9
print(f"  {a:04b}  ({a})")
print(f"| {b:04b}  ({b})")
print(f"= {a | b:04b}  ({a | b})")

0 | 0 = 0
0 | 1 = 1
1 | 0 = 1
1 | 1 = 1
  0100  (4)
| 1001  (9)
= 1101  (13)


### 3.3. Ba tính chất hay dùng của OR

- **`a | 0 = a`**: OR với `0` không đổi gì.
- **`a | a = a`**: OR với chính nó không đổi.
- **`a | b >= max(a, b)`**: OR chỉ **thêm** bit `1`, không bao giờ bớt, nên kết quả không nhỏ hơn mỗi số.
- **Giao hoán**: `a | b = b | a`; nên `nums[i] | nums[i+1]` không phụ thuộc thứ tự hai số.

In [4]:
a, b = 5, 11
print(a | 0, a | a, b | b)                # 5 5 11
print(a | b, max(a, b), (a | b) >= max(a, b))   # 15 11 True
print((a | b) == (b | a))                 # True

5 5 11
15 11 True
True


### 3.4. `|` khác `or` và khác `+`

- **`or` là toán tử logic:** `a or b` trả về `a` nếu `a` "đúng" (khác `0`), ngược lại trả về `b`. Nó **không** làm việc trên bit.
- **`+` là phép cộng:** có nhớ (carry). `3 + 1 = 4` nhưng `3 | 1 = 3` (vì bit thấp nhất đã là `1`).

Vì vậy với bài này bắt buộc dùng **`|`**.

In [5]:
print(8 or 4, "<- 'or' trả về 8, không phải 12")
print(8 | 4, "<- '|' ra 12")
print(0 or 7, "<- 0 là 'sai' nên trả về 7")
print(3 + 1, "vs", 3 | 1)                  # 4 vs 3
print(5 + 4, "vs", 5 | 4)                  # 9 vs 5

8 <- 'or' trả về 8, không phải 12
12 <- '|' ra 12
7 <- 0 là 'sai' nên trả về 7
4 vs 3
9 vs 5


### 3.5. Cặp phần tử kề nhau: `range(n - 1)`, `zip` và `pairwise`

Mảng `n` phần tử có `n - 1` cặp kề nhau `(nums[i], nums[i+1])` với `i` từ `0` đến `n - 2`, tức `range(n - 1)`. Ba cách liệt kê các cặp:

- `for i in range(len(nums) - 1)` rồi dùng `nums[i]`, `nums[i + 1]`;
- `zip(nums, nums[1:])`: ghép mảng với chính nó dịch một vị trí;
- `itertools.pairwise(nums)` (Python 3.10+): cho thẳng các cặp kề nhau.

In [6]:
nums = [8, 4, 2]
print(len(nums) - 1, "cặp")
print([(nums[i], nums[i + 1]) for i in range(len(nums) - 1)])
print(list(zip(nums, nums[1:])))
print(list(pairwise(nums)))

2 cặp
[(8, 4), (4, 2)]
[(8, 4), (4, 2)]
[(8, 4), (4, 2)]


### 3.6. List comprehension và `map`

- List comprehension: `[biểu_thức for phần_tử in dãy]` tạo danh sách mới. Ví dụ `[x * 2 for x in [1, 2, 3]]` là `[2, 4, 6]`.
- `map(hàm, dãy1, dãy2)` áp dụng `hàm` lên **từng cặp** phần tử song song và dừng khi dãy ngắn nhất hết. `operator.or_` là hàm tương ứng toán tử `|`.

In [7]:
print([x * 2 for x in [1, 2, 3]])                       # [2, 4, 6]
print([a | b for a, b in [(8, 4), (4, 2)]])             # [12, 6]
print(list(map(or_, [8, 4, 2], [4, 2])))                # [12, 6] : map dừng khi dãy ngắn hết

[2, 4, 6]
[12, 6]
[12, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [5, 4, 9, 11]` ta viết mỗi số bằng `4` bit rồi OR **từng cặp kề nhau**:

| cặp | `nums[i]` | `nums[i+1]` | OR theo bit | kết quả |
|---|---|---|---|---|
| `(5, 4)` | `0101` | `0100` | `0101` | `5` |
| `(4, 9)` | `0100` | `1001` | `1101` | `13` |
| `(9, 11)` | `1001` | `1011` | `1011` | `11` |

Mảng `4` phần tử cho `3` cặp nên `answer = [5, 13, 11]` có `3` phần tử.

Quan sát: **mỗi ô kết quả chỉ nhìn hai ô liền nhau**. Không có phụ thuộc giữa `answer[0]` và `answer[1]`, nên ta có thể tính theo bất kỳ thứ tự nào.

In [8]:
nums = [5, 4, 9, 11]
for i in range(len(nums) - 1):
    a, b = nums[i], nums[i + 1]
    print(f"{a:>2} = {a:04b}  |  {b:>2} = {b:04b}  ->  {a | b:04b} = {a | b}")

 5 = 0101  |   4 = 0100  ->  0101 = 5
 4 = 0100  |   9 = 1001  ->  1101 = 13
 9 = 1001  |  11 = 1011  ->  1011 = 11


## 5. Cách 1 — Brute Force / baseline: tự OR từng bit

**Ý tưởng.** Để hiểu `|` thực sự làm gì, ta tự cài đặt OR bằng cách xét từng bit: với mỗi vị trí bit `p` từ `0` đến `6` (đủ cho giá trị `<= 100 < 128`), lấy bit thứ `p` của `a` và của `b` bằng `(a >> p) & 1`; nếu **ít nhất một** bit là `1` thì cộng `1 << p` vào kết quả. Sau đó áp dụng cho mọi cặp kề nhau.

In [9]:
def or_by_bits(a: int, b: int) -> int:
    result = 0
    for p in range(8):                              # 8 bit đủ cho giá trị tới 255
        bit_a = (a >> p) & 1                        # bit thứ p của a
        bit_b = (b >> p) & 1                        # bit thứ p của b
        if bit_a == 1 or bit_b == 1:
            result += 1 << p                        # bật bit thứ p của kết quả
    return result


def or_array_bits(nums: List[int]) -> List[int]:
    answer = []
    for i in range(len(nums) - 1):
        answer.append(or_by_bits(nums[i], nums[i + 1]))
    return answer


print(or_by_bits(4, 9), 4 | 9)                      # 13 13
for nums, expected in examples:
    print(nums, "->", or_array_bits(nums), "(kỳ vọng", expected, ")")

13 13
[1, 3, 7, 15] -> [3, 7, 15] (kỳ vọng [3, 7, 15] )
[8, 4, 2] -> [12, 6] (kỳ vọng [12, 6] )
[5, 4, 9, 11] -> [5, 13, 11] (kỳ vọng [5, 13, 11] )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`, `B = 8` là số bit xét (hằng số). Mỗi cặp tốn `O(B)` ⇒ **Time `O(n * B) = O(n)`**, **Space `O(n)`** cho mảng kết quả (`O(1)` phụ ngoài kết quả).

**Nút thắt:** ta tự làm việc mà CPU đã có sẵn một lệnh `|` làm trong một bước. Cải tiến: dùng thẳng toán tử `|`.

## 6. Ý tưởng thuật toán / bước đột phá

**Dùng thẳng `|` trên từng cặp kề nhau.** Vì `answer[i] = nums[i] | nums[i + 1]` và hai chỉ số này luôn tồn tại với `i` từ `0` đến `n - 2`:

1. Tạo `answer` rỗng.
2. Với `i` trong `range(len(nums) - 1)`: `answer.append(nums[i] | nums[i + 1])`.
3. Trả về `answer`.

Điểm cần nhớ: **độ dài kết quả là `n - 1`** — vòng lặp dừng ở `n - 2`, không phải `n - 1`.

In [10]:
nums = [8, 4, 2]
answer = []
for i in range(len(nums) - 1):
    value = nums[i] | nums[i + 1]
    answer.append(value)
    print(f"i={i}: nums[{i}] | nums[{i + 1}] = {nums[i]} | {nums[i + 1]} = {value} -> answer = {answer}")
print("độ dài:", len(answer), "= len(nums) - 1 =", len(nums) - 1)

i=0: nums[0] | nums[1] = 8 | 4 = 12 -> answer = [12]
i=1: nums[1] | nums[2] = 4 | 2 = 6 -> answer = [12, 6]
độ dài: 2 = len(nums) - 1 = 2


## 7. Các cách tiếp theo

### Cách 2 — Một vòng lặp, `append`

**Giải thích từng đoạn.**

1. `answer = []`: danh sách kết quả, bắt đầu rỗng.
2. `for i in range(len(nums) - 1)`: mỗi `i` là một cặp kề nhau.
3. `answer.append(nums[i] | nums[i + 1])`: thêm kết quả OR của cặp.
4. `return answer`.

In [11]:
def or_array_loop(nums: List[int]) -> List[int]:
    answer = []
    for i in range(len(nums) - 1):
        answer.append(nums[i] | nums[i + 1])
    return answer


for nums, expected in examples:
    print(nums, "->", or_array_loop(nums), "(kỳ vọng", expected, ")")

[1, 3, 7, 15] -> [3, 7, 15] (kỳ vọng [3, 7, 15] )
[8, 4, 2] -> [12, 6] (kỳ vọng [12, 6] )
[5, 4, 9, 11] -> [5, 13, 11] (kỳ vọng [5, 13, 11] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionLoop:
    def orArray(self, nums: List[int]) -> List[int]:
        answer = []
        for i in range(len(nums) - 1):
            answer.append(nums[i] | nums[i + 1])
        return answer


print(SolutionLoop().orArray([8, 4, 2]))        # [12, 6]

[12, 6]


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** cho mảng kết quả (không tính vào bộ nhớ phụ).

### Cách 3 — Cấp phát sẵn `answer = [0] * (n - 1)` rồi gán theo chỉ số

**Giải thích.** Biết trước độ dài kết quả là `n - 1`, ta tạo sẵn `answer = [0] * (n - 1)` rồi `answer[i] = nums[i] | nums[i + 1]`. Cách này giống ngôn ngữ có mảng cố định (Java/C++), tránh `append`, và cho thấy rõ **`answer[i]` ứng với cặp `(i, i + 1)`**.

In [13]:
def or_array_prealloc(nums: List[int]) -> List[int]:
    n = len(nums)
    answer = [0] * (n - 1)                      # n - 1 ô
    for i in range(n - 1):
        answer[i] = nums[i] | nums[i + 1]
    return answer


for nums, expected in examples:
    print(nums, "->", or_array_prealloc(nums), "(kỳ vọng", expected, ")")

[1, 3, 7, 15] -> [3, 7, 15] (kỳ vọng [3, 7, 15] )
[8, 4, 2] -> [12, 6] (kỳ vọng [12, 6] )
[5, 4, 9, 11] -> [5, 13, 11] (kỳ vọng [5, 13, 11] )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionPrealloc:
    def orArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        answer = [0] * (n - 1)
        for i in range(n - 1):
            answer[i] = nums[i] | nums[i + 1]
        return answer


print(SolutionPrealloc().orArray([5, 4, 9, 11]))        # [5, 13, 11]

[5, 13, 11]


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (mảng kết quả).

### Cách 4 — `zip(nums, nums[1:])` + list comprehension

**Giải thích.** `zip(nums, nums[1:])` ghép mỗi phần tử với phần tử đứng sau nó, cho đúng `n - 1` cặp. Mỗi cặp `(a, b)` ta tính `a | b` trong một list comprehension: `[a | b for a, b in zip(nums, nums[1:])]`. Không cần chỉ số nên khó viết sai `IndexError`.

In [15]:
def or_array_zip(nums: List[int]) -> List[int]:
    return [a | b for a, b in zip(nums, nums[1:])]


for nums, expected in examples:
    print(nums, "->", or_array_zip(nums), "(kỳ vọng", expected, ")")

[1, 3, 7, 15] -> [3, 7, 15] (kỳ vọng [3, 7, 15] )
[8, 4, 2] -> [12, 6] (kỳ vọng [12, 6] )
[5, 4, 9, 11] -> [5, 13, 11] (kỳ vọng [5, 13, 11] )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionZip:
    def orArray(self, nums: List[int]) -> List[int]:
        return [a | b for a, b in zip(nums, nums[1:])]


print(SolutionZip().orArray([1, 3, 7, 15]))     # [3, 7, 15]

[3, 7, 15]


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (cả `nums[1:]` lẫn kết quả; phần phụ `O(n)` do bản sao `nums[1:]`).

### Cách 5 — `map(or_, nums, nums[1:])` hoặc `pairwise`

**Giải thích.** `operator.or_(a, b)` chính là `a | b`. `map(or_, nums, nums[1:])` áp hàm lên từng cặp song song và tự dừng khi `nums[1:]` hết — cho `n - 1` kết quả. Bản khác dùng `itertools.pairwise` (Python 3.10+) để không tạo `nums[1:]`: `[a | b for a, b in pairwise(nums)]`.

In [17]:
def or_array_map(nums: List[int]) -> List[int]:
    return list(map(or_, nums, nums[1:]))


def or_array_pairwise(nums: List[int]) -> List[int]:
    return [a | b for a, b in pairwise(nums)]


for nums, expected in examples:
    print(nums, "->", or_array_map(nums), or_array_pairwise(nums), "(kỳ vọng", expected, ")")

[1, 3, 7, 15] -> [3, 7, 15] [3, 7, 15] (kỳ vọng [3, 7, 15] )
[8, 4, 2] -> [12, 6] [12, 6] (kỳ vọng [12, 6] )
[5, 4, 9, 11] -> [5, 13, 11] [5, 13, 11] (kỳ vọng [5, 13, 11] )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionMap:
    def orArray(self, nums: List[int]) -> List[int]:
        return list(map(or_, nums, nums[1:]))


print(SolutionMap().orArray([8, 4, 2]))         # [12, 6]

[12, 6]


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (kết quả; `pairwise` không tạo bản sao nên phần phụ `O(1)`).

## 8. TRACE từng bước

Ta trace Cách 2 (vòng lặp `append`) trên `nums = [5, 4, 9, 11]` có in cả dạng nhị phân, và Cách 4 (`zip`) trên `nums = [8, 4, 2]`.

In [19]:
def trace_loop(nums: List[int]) -> List[int]:
    print(f"nums = {nums}, n = {len(nums)} -> kết quả có {len(nums) - 1} phần tử")
    answer = []
    for i in range(len(nums) - 1):
        a, b = nums[i], nums[i + 1]
        value = a | b
        answer.append(value)
        print(f"  i={i}: {a:>2} ({a:04b}) | {b:>2} ({b:04b}) = {value:>2} ({value:04b}) -> answer = {answer}")
    return answer


assert trace_loop([5, 4, 9, 11]) == [5, 13, 11]
assert trace_loop([1, 3, 7, 15]) == [3, 7, 15]

nums = [5, 4, 9, 11], n = 4 -> kết quả có 3 phần tử
  i=0:  5 (0101) |  4 (0100) =  5 (0101) -> answer = [5]
  i=1:  4 (0100) |  9 (1001) = 13 (1101) -> answer = [5, 13]
  i=2:  9 (1001) | 11 (1011) = 11 (1011) -> answer = [5, 13, 11]
nums = [1, 3, 7, 15], n = 4 -> kết quả có 3 phần tử
  i=0:  1 (0001) |  3 (0011) =  3 (0011) -> answer = [3]
  i=1:  3 (0011) |  7 (0111) =  7 (0111) -> answer = [3, 7]
  i=2:  7 (0111) | 15 (1111) = 15 (1111) -> answer = [3, 7, 15]


In [20]:
def trace_zip(nums: List[int]) -> List[int]:
    shifted = nums[1:]
    print(f"nums    = {nums}")
    print(f"nums[1:] = {shifted}")
    pairs = list(zip(nums, shifted))
    print(f"zip     = {pairs}")
    answer = []
    for a, b in pairs:
        answer.append(a | b)
        print(f"  ({a}, {b}) -> {a} | {b} = {a | b}")
    return answer


assert trace_zip([8, 4, 2]) == [12, 6]

nums    = [8, 4, 2]
nums[1:] = [4, 2]
zip     = [(8, 4), (4, 2)]
  (8, 4) -> 8 | 4 = 12
  (4, 2) -> 4 | 2 = 6


Bảng trace Cách 2 với `nums = [5, 4, 9, 11]`:

| i | `nums[i]` | `nums[i+1]` | nhị phân | `answer` sau bước |
|---|---|---|---|---|
| 0 | 5 | 4 | `0101 \| 0100 = 0101` | `[5]` |
| 1 | 4 | 9 | `0100 \| 1001 = 1101` | `[5, 13]` |
| 2 | 9 | 11 | `1001 \| 1011 = 1011` | `[5, 13, 11]` |

Vòng lặp dừng ở `i = 2 = n - 2`. Đáp án `[5, 13, 11]`.

Bảng trace Cách 4 với `nums = [8, 4, 2]`: `nums[1:] = [4, 2]`; `zip` cho `(8, 4)` và `(4, 2)` (số cặp `= min(3, 2) = 2`); kết quả `[8 | 4, 4 | 2] = [12, 6]`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (vòng lặp `append`): rõ ràng, không phụ thuộc phiên bản Python.

In [21]:
class Solution:
    def orArray(self, nums: List[int]) -> List[int]:
        answer = []
        for i in range(len(nums) - 1):                 # n - 1 cặp kề nhau
            answer.append(nums[i] | nums[i + 1])       # OR theo bit của cặp (i, i + 1)
        return answer


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.orArray(nums), "(kỳ vọng", expected, ")")

[1, 3, 7, 15] -> [3, 7, 15] (kỳ vọng [3, 7, 15] )
[8, 4, 2] -> [12, 6] (kỳ vọng [12, 6] )
[5, 4, 9, 11] -> [5, 13, 11] (kỳ vọng [5, 13, 11] )


## 10. Vì sao thuật toán đúng

- **Đúng định nghĩa:** `answer[i] = nums[i] | nums[i + 1]` với `i = 0, 1, ..., n - 2`. Vòng lặp tính đúng công thức đó cho từng `i` theo thứ tự, và `append` đặt kết quả của `i` vào vị trí `i` của `answer`.
- **Đúng độ dài:** vòng lặp chạy `n - 1` lần (`i = 0..n - 2`) nên `answer` có đúng `n - 1` phần tử, khớp đề. Chỉ số lớn nhất được đọc là `nums[n - 1]` (ở `i = n - 2`), luôn hợp lệ.
- **Không phụ thuộc lẫn nhau:** mỗi ô chỉ phụ thuộc vào hai phần tử của `nums` (ta **không sửa** `nums` trong khi tính), nên thứ tự tính không ảnh hưởng.
- **Phép `|` đúng bản chất:** bit kết quả bằng `1` nếu ít nhất một trong hai bit là `1` — khớp với Cách 1 (`or_by_bits`), và với đồng nhất thức `a | b = a + b - (a & b)` mà ta dùng làm oracle khi test.
- **Mảng có `n >= 2`:** đề đảm bảo có ít nhất một cặp nên `answer` không rỗng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 100`).

| Cách | Time | Space (không tính kết quả) |
|---|---|---|
| 1. OR từng bit tự cài | `O(n * B)` với `B = 8` ⇒ `O(n)` | `O(1)` |
| 2. Vòng lặp `append` | `O(n)` | `O(1)` |
| 3. Cấp phát sẵn `[0] * (n - 1)` | `O(n)` | `O(1)` |
| 4. `zip(nums, nums[1:])` | `O(n)` | `O(n)` (bản sao `nums[1:]`) |
| 5. `map(or_, ...)` / `pairwise` | `O(n)` | `O(n)` / `O(1)` |

Kích thước đầu ra là `n - 1` nên **không thể tốt hơn `O(n)`**.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [22]:
sol = Solution()
impls = {
    "bits": or_array_bits,
    "loop": or_array_loop,
    "prealloc": or_array_prealloc,
    "zip": or_array_zip,
    "map": or_array_map,
    "pairwise": or_array_pairwise,
    "leetcode": sol.orArray,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [23]:
check_all([0, 0], [0], "hai số 0")
check_all([0, 5], [5], "OR với 0 không đổi")
check_all([7, 7], [7], "hai số bằng nhau")
check_all([100, 100], [100], "giá trị lớn nhất, bằng nhau")
check_all([1, 2], [3], "hai bit khác nhau: 1 | 2 = 3")
check_all([64, 32, 16, 8, 4, 2, 1], [96, 48, 24, 12, 6, 3], "dãy lũy thừa của 2 giảm dần")
check_all([1, 2, 4, 8, 16, 32, 64], [3, 6, 12, 24, 48, 96], "dãy lũy thừa của 2 tăng dần")
check_all([100, 0, 100], [100, 100], "đệm số 0")
check_all([127 & 100, 3, 0], [(100 & 127) | 3, 3], "kết hợp phép & để tạo số")
check_all([3, 5, 6], [7, 7], "3|5 = 7, 5|6 = 7")
check_all([10, 20, 30, 40, 50], [10 | 20, 20 | 30, 30 | 40, 40 | 50], "bội của 10")
check_all(list(range(1, 101)), [i | (i + 1) for i in range(1, 100)], "1..100: n = 100, kết quả 99 phần tử")

PASS hai số 0
PASS OR với 0 không đổi
PASS hai số bằng nhau
PASS giá trị lớn nhất, bằng nhau
PASS hai bit khác nhau: 1 | 2 = 3
PASS dãy lũy thừa của 2 giảm dần
PASS dãy lũy thừa của 2 tăng dần
PASS đệm số 0
PASS kết hợp phép & để tạo số
PASS 3|5 = 7, 5|6 = 7
PASS bội của 10
PASS 1..100: n = 100, kết quả 99 phần tử


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) dùng **`or`** thay cho `|`; (b) dùng **`+`** thay cho `|`.

In [24]:
def wrong_logical_or(nums):
    return [a or b for a, b in zip(nums, nums[1:])]          # SAI: or logic


def wrong_plus(nums):
    return [a + b for a, b in zip(nums, nums[1:])]           # SAI: cộng


print(wrong_logical_or([8, 4, 2]), "!=", or_array_loop([8, 4, 2]))        # [8, 4] so với [12, 6]
print(wrong_plus([5, 4, 9, 11]), "!=", or_array_loop([5, 4, 9, 11]))       # [9, 13, 20] so với [5, 13, 11]
assert wrong_logical_or([8, 4, 2]) != or_array_loop([8, 4, 2])
assert wrong_plus([5, 4, 9, 11]) != or_array_loop([5, 4, 9, 11])

[8, 4] != [12, 6]
[9, 13, 20] != [5, 13, 11]


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: độ dài `2..100`, giá trị `0..100`. Ta đối chiếu mọi cách với một **oracle** độc lập dùng đồng nhất thức `a | b = a + b - (a & b)` (không dùng `|`). Thêm các **thuộc tính**: độ dài kết quả luôn `n - 1`; mỗi phần tử kết quả `>= max` của hai số tương ứng; `nums` không bị sửa. **Vét cạn** mọi mảng độ dài `2` và `3` với giá trị `0..9`.

In [25]:
def oracle(nums):
    return [a + b - (a & b) for a, b in zip(nums, nums[1:])]


rng = random.Random(3173)
for trial in range(4000):
    n = rng.randint(2, 100)
    hi = rng.choice([1, 3, 7, 15, 100])
    nums = [rng.randint(0, hi) for _ in range(n)]
    assert 2 <= n <= 100 and all(0 <= x <= 100 for x in nums)
    ref = oracle(nums)
    backup = list(nums)
    for key, fn in impls.items():
        res = fn(nums)
        assert res == ref, (key, nums)
        assert len(res) == n - 1
        assert all(r >= max(a, b) for r, a, b in zip(res, nums, nums[1:]))
        assert nums == backup, ("hàm đã sửa đầu vào", key)
print("PASS 4000 mảng ngẫu nhiên")

count = 0
for n in (2, 3):
    for combo in itertools.product(range(10), repeat=n):
        nums = list(combo)
        ref = oracle(nums)
        for key, fn in impls.items():
            assert fn(list(nums)) == ref, (key, nums)
        count += 1
print("PASS vét cạn", count, "mảng nhỏ")

PASS 4000 mảng ngẫu nhiên
PASS vét cạn 1100 mảng nhỏ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `or` (logic) thay cho `|` (bit)

In [26]:
def sai_dung_or(nums):
    answer = []
    for i in range(len(nums) - 1):
        answer.append(nums[i] or nums[i + 1])          # SAI: 'or' không làm việc trên bit
    return answer


print("sai:", sai_dung_or([8, 4, 2]), "| đúng:", Solution().orArray([8, 4, 2]))
print("sai:", sai_dung_or([0, 5, 0]), "| đúng:", Solution().orArray([0, 5, 0]))

sai: [8, 4] | đúng: [12, 6]
sai: [5, 5] | đúng: [5, 5]


**Vì sao sai:** `a or b` trả về `a` nếu `a` khác `0`, ngược lại trả về `b` — chỉ chọn một trong hai số. Ví dụ `8 or 4 = 8` chứ không phải `12`. **Cách sửa:** dùng `|` cho OR theo bit (`or` dành cho điều kiện `if`).

### Lỗi 2: Dùng `+` thay cho `|`

In [27]:
def sai_dung_cong(nums):
    return [nums[i] + nums[i + 1] for i in range(len(nums) - 1)]      # SAI: cộng


print("sai:", sai_dung_cong([5, 4, 9, 11]), "| đúng:", Solution().orArray([5, 4, 9, 11]))
print("sai:", sai_dung_cong([8, 4, 2]), "| đúng:", Solution().orArray([8, 4, 2]), "(trùng vì các số không chung bit)")

sai: [9, 13, 20] | đúng: [5, 13, 11]
sai: [12, 6] | đúng: [12, 6] (trùng vì các số không chung bit)


**Vì sao sai:** cộng và OR chỉ trùng khi hai số **không có bit chung** (`8` và `4` không chung bit nên `8 + 4 = 8 | 4 = 12`). Nếu có bit chung (`5 = 101`, `4 = 100` chung bit thứ `2`) thì cộng sinh nhớ và **khác** OR. Ví dụ 2 trùng "ngẫu nhiên", ví dụ 3 mới lộ lỗi. **Cách sửa:** dùng `|`.

### Lỗi 3: Kết quả dài `n` / `IndexError` vì `range(len(nums))`

In [28]:
def sai_range_day_du(nums):
    answer = []
    for i in range(len(nums)):                          # SAI: i chạy tới n - 1
        answer.append(nums[i] | nums[i + 1])
    return answer


try:
    sai_range_day_du([8, 4, 2])
except IndexError as e:
    print("IndexError:", e)


def sai_them_phan_tu_cuoi(nums):
    answer = [nums[i] | nums[i + 1] for i in range(len(nums) - 1)]
    answer.append(nums[-1])                              # SAI: đề chỉ cần n - 1 phần tử
    return answer


print("sai:", sai_them_phan_tu_cuoi([8, 4, 2]), "| đúng:", Solution().orArray([8, 4, 2]))

IndexError: list index out of range
sai: [12, 6, 2] | đúng: [12, 6]


**Vì sao sai:** có `n` phần tử thì chỉ có `n - 1` cặp kề nhau; ở `i = n - 1` không có `nums[i + 1]`. Cố "bù" thêm một phần tử cuối làm kết quả dài `n`, sai độ dài. **Cách sửa:** `range(len(nums) - 1)` và không thêm gì sau đó.

### Lỗi 4: Sửa mảng tại chỗ theo hướng sai (từ phải sang trái)

In [29]:
def sai_tai_cho_phai_sang_trai(nums):
    nums = list(nums)
    for i in range(len(nums) - 2, -1, -1):             # SAI: nums[i + 1] đã bị ghi đè ở bước trước
        nums[i] = nums[i] | nums[i + 1]
    return nums[:-1]


def dung_tai_cho_trai_sang_phai(nums):
    nums = list(nums)
    for i in range(len(nums) - 1):                      # ĐÚNG: nums[i + 1] còn nguyên khi dùng
        nums[i] = nums[i] | nums[i + 1]
    return nums[:-1]


print("sai :", sai_tai_cho_phai_sang_trai([8, 4, 2]), "| đúng:", Solution().orArray([8, 4, 2]))
print("trái -> phải:", dung_tai_cho_trai_sang_phai([8, 4, 2]))
print("sai :", sai_tai_cho_phai_sang_trai([1, 2, 4, 8]), "| đúng:", Solution().orArray([1, 2, 4, 8]))

sai : [14, 6] | đúng: [12, 6]
trái -> phải: [12, 6]
sai : [15, 14, 12] | đúng: [3, 6, 12]


**Vì sao sai:** khi sửa tại chỗ, ô `nums[i]` được ghi đè bằng `nums[i] | nums[i + 1]`. Nếu đi từ phải sang trái, `nums[i + 1]` **đã bị ghi đè** (đã chứa OR với ô bên phải), nên giá trị gộp thêm các bit của ô xa hơn — ví dụ `[8, 4, 2]` cho ra `[14, 6]` thay vì `[12, 6]`. Đi **từ trái sang phải** thì mỗi `nums[i + 1]` còn nguyên khi được đọc, nên đúng. **Cách tốt nhất:** không sửa đầu vào, tạo mảng kết quả mới.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Với mỗi cặp phần tử kề nhau, tính một giá trị**" ⇒ kết quả dài `n - 1`; dùng `zip(nums, nums[1:])` hoặc `pairwise`.
- OR / AND / XOR theo bit ⇒ toán tử **`|`, `&`, `^`** (không phải `or`, `and`, `!=`).
- Cần hiểu bit ⇒ in `f"{x:08b}"` để nhìn từng bit.
- Đồng nhất thức hữu ích: `a | b = a + b - (a & b)`; `a ^ b = (a | b) - (a & b)`; `a | b >= max(a, b)`.
- Mỗi ô kết quả độc lập ⇒ không cần cấu trúc dữ liệu phụ; dễ viết dạng comprehension.
- Luôn kiểm tra **độ dài kết quả** và ca nhỏ nhất `n = 2`.
- Các bài cùng họ: *Adjacent Increasing Subarrays*, *Special Array I (3151)*, *Number of Even and Odd Bits*, *Neighboring Bitwise XOR (2683)*.

## 15. Biến thể của bài

1. **AND hoặc XOR của cặp kề nhau:** thay `|` bằng `&` hoặc `^`.
2. **Cửa sổ ba phần tử:** `answer[i] = nums[i] | nums[i + 1] | nums[i + 2]`.
3. **OR của mọi phần tử** (`reduce(or_, nums)`) và **OR của mọi mảng con** (nâng cao).
4. **Vòng tròn:** coi phần tử cuối kề phần tử đầu, kết quả dài `n`.
5. **Giải ngược:** cho `answer`, tìm một `nums` có thể (nhiều đáp án!) — liên quan *Neighboring Bitwise XOR (2683)*.

In [30]:
from functools import reduce


def and_xor_adjacent(nums):
    return [a & b for a, b in pairwise(nums)], [a ^ b for a, b in pairwise(nums)]


def or_window3(nums):
    return [nums[i] | nums[i + 1] | nums[i + 2] for i in range(len(nums) - 2)]


def or_circular(nums):
    n = len(nums)
    return [nums[i] | nums[(i + 1) % n] for i in range(n)]


def or_all(nums):
    return reduce(or_, nums, 0)


print(and_xor_adjacent([5, 4, 9, 11]))     # ([4, 0, 9], [1, 13, 2])
print(or_window3([1, 2, 4, 8]))            # [7, 14]
print(or_circular([8, 4, 2]))              # [12, 6, 10]
print(or_all([1, 2, 4, 8]))                # 15
assert and_xor_adjacent([5, 4, 9, 11]) == ([4, 0, 9], [1, 13, 2])
assert or_window3([1, 2, 4, 8]) == [7, 14]
assert or_circular([8, 4, 2]) == [12, 6, 10]
assert or_all([1, 2, 4, 8]) == 15
assert or_all([]) == 0

([4, 0, 9], [1, 13, 2])
[7, 14]
[12, 6, 10]
15


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. OR từng bit tự cài | `(a >> p) & 1`, bật bit `1 << p` | `O(n * B)` | `O(1)` | học cơ chế của `\|` |
| 2. Vòng lặp `append` | `answer.append(nums[i] \| nums[i + 1])` | `O(n)` | `O(1)` phụ | **nên nộp**: rõ ràng nhất |
| 3. Cấp phát sẵn | `[0] * (n - 1)` rồi gán `answer[i]` | `O(n)` | `O(1)` phụ | quen tư duy mảng cố định |
| 4. `zip` + comprehension | `[a \| b for a, b in zip(nums, nums[1:])]` | `O(n)` | `O(n)` phụ | viết ngắn, không lo `IndexError` |
| 5. `map(or_, ...)` / `pairwise` | hàm `or_` trên từng cặp | `O(n)` | `O(n)` / `O(1)` phụ | phong cách hàm, `pairwise` tiết kiệm bộ nhớ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `6 | 3`, `6 & 3`, `f"{10:04b}"`, `0 or 9`, `Solution().orArray([2, 5])`.

**Bài 2 (tính tay).** Với `nums = [12, 5, 3, 10]`: viết từng số bằng `4` bit, OR từng cặp kề nhau và cho biết `answer`.

**Bài 3 (điền TODO).** Hoàn thành `or_array_todo(nums)` theo Cách 2.

In [31]:
def or_array_todo(nums):
    # TODO 1: answer = []
    # TODO 2: duyệt i từ 0 đến len(nums) - 2: thêm nums[i] | nums[i + 1] vào answer
    # TODO 3: trả về answer
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `or_array_bitstrings(nums)`: đổi mỗi số thành chuỗi nhị phân độ dài `8` (`f"{x:08b}"`), OR từng bit **bằng chuỗi** (ký tự `'1'` nếu một trong hai là `'1'`), rồi đổi ngược về số bằng `int(chuoi, 2)`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `and_array(nums)` trả `answer[i] = nums[i] & nums[i + 1]`. (b) Viết `or_array_k(nums, k)` với `answer[i] = OR của k phần tử liên tiếp nums[i..i+k-1]` (độ dài `n - k + 1`; với `k = 2` phải trùng đề).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6 = 110`, `3 = 011` nên `6 | 3 = 111 = 7`, `6 & 3 = 010 = 2`; `f"{10:04b}"` là `'1010'`; `0 or 9 = 9`; `[2, 5]` ra `[7]` (`010 | 101 = 111`). Bài 3: `answer.append(nums[i] | nums[i + 1])` trong `range(len(nums) - 1)`.
- **Mức 2 (vừa):** Bài 2: `12 = 1100`, `5 = 0101`, `3 = 0011`, `10 = 1010`; `1100 | 0101 = 1101 = 13`; `0101 | 0011 = 0111 = 7`; `0011 | 1010 = 1011 = 11`; `answer = [13, 7, 11]`. Bài 4: duyệt cặp ký tự bằng `zip(sa, sb)`.
- **Mức 3 (sâu):** Bài 5(a): giống đề nhưng `&`. 5(b): `[reduce(or_, nums[i:i + k]) for i in range(len(nums) - k + 1)]`; kiểm tra `k = 2` trùng `Solution().orArray`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert 6 | 3 == 7
assert 6 & 3 == 2
assert f"{10:04b}" == "1010"
assert (0 or 9) == 9
assert Solution().orArray([2, 5]) == [7]

# Bài 2
assert Solution().orArray([12, 5, 3, 10]) == [13, 7, 11]
assert [f"{x:04b}" for x in [12, 5, 3, 10]] == ["1100", "0101", "0011", "1010"]


# Bài 3
def or_array_todo_ans(nums):
    answer = []
    for i in range(len(nums) - 1):
        answer.append(nums[i] | nums[i + 1])
    return answer


for nums, expected in examples:
    assert or_array_todo_ans(nums) == expected


# Bài 4
def or_array_bitstrings(nums):
    answer = []
    for a, b in zip(nums, nums[1:]):
        sa, sb = f"{a:08b}", f"{b:08b}"
        merged = "".join("1" if x == "1" or y == "1" else "0" for x, y in zip(sa, sb))
        answer.append(int(merged, 2))
    return answer


for nums, expected in examples:
    assert or_array_bitstrings(nums) == expected


# Bài 5(a)
def and_array(nums):
    return [a & b for a, b in zip(nums, nums[1:])]


assert and_array([5, 4, 9, 11]) == [4, 0, 9]
assert and_array([7, 7]) == [7]


# Bài 5(b)
def or_array_k(nums, k):
    return [reduce(or_, nums[i:i + k]) for i in range(len(nums) - k + 1)]


assert or_array_k([1, 2, 4, 8], 3) == [7, 14]
assert or_array_k([1, 2, 4, 8], 1) == [1, 2, 4, 8]
for nums, expected in examples:
    assert or_array_k(nums, 2) == expected
rng2 = random.Random(7)
for _ in range(300):
    arr = [rng2.randint(0, 100) for _ in range(rng2.randint(2, 100))]
    assert or_array_bitstrings(arr) == Solution().orArray(arr) == or_array_k(arr, 2)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (kết quả cho mỗi cặp kề nhau, độ dài `n - 1`)

```python
class Solution:
    def orArray(self, nums: List[int]) -> List[int]:
        answer = []
        for i in range(len(nums) - 1):
            answer.append(nums[i] | nums[i + 1])
        return answer
```

**Checklist**

- [ ] Dùng toán tử **`|`** (OR bit), không dùng `or` và không dùng `+`?
- [ ] Vòng lặp là `range(len(nums) - 1)` và kết quả dài đúng `n - 1`?
- [ ] Không sửa mảng đầu vào (hoặc nếu sửa tại chỗ thì đi từ trái sang phải)?
- [ ] Đã thử `[1,3,7,15]` (`[3,7,15]`), `[8,4,2]` (`[12,6]`), `[5,4,9,11]` (`[5,13,11]`), và `n = 2`?
- [ ] Đã đối chiếu lại đề gốc nếu có tài khoản Premium (đây là bài Premium)?
- [ ] Skeleton đúng `class Solution` / `def orArray(self, nums: List[int]) -> List[int]`?